# Phase 3 — Training
## Brain Tumour MRI Classification
====================================================================

Train the Phase 2 model on the Phase 1 split. The test set is not touched here
and is not touched until Phase 5.

Two controls run before any training curve is believed, and the class balancing
left undecided in Phase 1 is decided here by measurement.

In [1]:
import sys, time
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))

import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import numpy as np
import torch

from src import config, data, engine, manifest, metrics, splits, viz
from src.config import (BATCH_SIZE, CHENG_DIR, CKPT_PATH, CLASSES,
                        DEVICE, EPOCHS, IMG_SIZE, LR, PATIENCE, SEED, WD)
from src.model import BrainTumourNet, count_parameters, receptive_field

D = splits.build_dataset(CHENG_DIR)
images, labels, groups = D["images"], D["labels"], D["groups"]
S = splits.build_splits(images, labels, groups)
train_idx, val_idx, test_idx = S["train_idx"], S["val_idx"], S["test_idx"]

# Normalisation from the TRAINING split only. These two constants are learned
# from data, so computing them across validation or test would let information
# about held-out scans reach the training pipeline -- leakage of a quieter kind
# than duplicate images, and just as real.
MEAN, STD = data.compute_stats(images, train_idx)

assert manifest.read() is not None, "run Phase 1 first -- no run manifest"
assert manifest.read()["split_hash"] == S["split_hash"], \
    "split does not match the run manifest -- re-run Phase 1"

print(f"device {DEVICE}   train {len(train_idx)}   val {len(val_idx)}   "
      f"test {len(test_idx)}  (test is not touched in this phase)")
print(f"run {manifest.current_hash()}   split {S['split_hash']}")
print(f"norm  mean {MEAN:.4f}  std {STD:.4f}   from the training split only")


device cuda   train 2099   val 352   test 613  (test is not touched in this phase)
run b9dcd147c12e   split 02b0799a48a71b69
norm  mean 0.1990  std 0.1585   from the training split only


In [ ]:
# 1. TWO CONTROLS BEFORE ANY CURVE IS BELIEVED
"""
A training curve that goes down looks the same whether the pipeline is correct
or subtly broken. These two runs distinguish those cases, and both are cheap.

The first checks the model can learn at all: given a handful of images per
class and no augmentation, it must reach near-perfect training accuracy by
memorising them. Sixteen per class at batch size 16 divides exactly, so no
image is dropped -- an earlier version took ten per class at the default batch
size, which happened to leave one usable batch at four classes and none at
three, and failed with a division by zero rather than anything legible.
A model that cannot overfit forty images has something wrong with its gradients,
its learning rate or its head, and no amount of tuning on the full set will fix
that.

The second checks the model is not learning something it should not: with the
labels shuffled, the relationship between image and label is destroyed, so
validation accuracy must sit at the no-information level. If it scores above
that, the split leaks and every subsequent number is meaningless.
"""
PER_CLASS, MEM_BATCH = 16, 16
small = np.concatenate([np.random.RandomState(0).choice(
    train_idx[labels[train_idx] == c], PER_CLASS, replace=False)
    for c in range(len(CLASSES))])

h_mem = engine.run_experiment(images, labels, small, small, MEAN, STD,
                              deep=True, augment=False, epochs=40,
                              batch_size=MEM_BATCH, verbose=False)
print(f"control 1  memorise {len(small)} images   "
      f"train acc {h_mem['train_acc'][-1]:.4f}   (expect ~1.00)")

shuffled = labels.copy()
rng = np.random.RandomState(0)
shuffled[train_idx] = rng.permutation(shuffled[train_idx])
shuffled[val_idx]   = rng.permutation(shuffled[val_idx])
h_shuf = engine.run_experiment(images, shuffled, train_idx, val_idx, MEAN, STD,
                               deep=True, augment=False, epochs=8, verbose=False)
MAJORITY = np.bincount(labels[val_idx]).max() / len(val_idx)
SE = np.sqrt(MAJORITY * (1 - MAJORITY) / len(val_idx))
SHUF_CEIL = MAJORITY + 3 * SE
shuf_acc = max(h_shuf["val_acc"])
ys, ps, _ = metrics.predict(h_shuf["model"], h_shuf["val_loader"])
shuf_f1 = metrics.macro_f1(ys, ps, len(CLASSES))

print(f"control 2  shuffled labels       best val acc {shuf_acc:.4f}   "
      f"macro F1 {shuf_f1:.4f}")
print(f"           no-information level  majority class {MAJORITY:.4f}, "
      f"1/K {1/len(CLASSES):.4f}")
print(f"           ceiling               {SHUF_CEIL:.4f}  "
      f"(majority + 3 SE, SE = {SE:.4f})")

control 1  memorise 48 images   train acc 1.0000   (expect ~1.00)
control 2  shuffled labels       best val acc 0.4631   macro F1 0.2110
           no-information level  majority class 0.4631, 1/K 0.3333
           ceiling               0.5428  (majority + 3 SE, SE = 0.0266)


In [ ]:
# 2. THE CLASS BALANCE DECISION, MEASURED
"""
Three corrections are implemented and exactly one may be used, since they all
correct the same imbalance and applying two would correct it twice:

  weights      scale the loss by inverse class frequency
  sampler      draw minority images more often, so each epoch is balanced
  undersample  truncate every class to the smallest, discarding 31% of training

The comparison is on macro F1, not accuracy. Accuracy on an uneven validation
set is a weighted average that hides exactly the class the correction is meant
to help, so selecting on it would be measuring the wrong thing.

Short runs, because this is a comparison between configurations rather than a
final model, and every configuration gets the identical budget.
"""
BAL_EPOCHS = 20
bal_rows = []
for choice in (None, "weights", "sampler", "undersample"):
    t0 = time.time()
    h = engine.run_experiment(images, labels, train_idx, val_idx, MEAN, STD,
                              deep=True, augment=True, balance=choice,
                              epochs=BAL_EPOCHS, verbose=False)
    y, p, _ = metrics.predict(h["model"], h["val_loader"])
    f1 = metrics.macro_f1(y, p, len(CLASSES))
    per = [float((p[y == c] == c).mean()) for c in range(len(CLASSES))]
    bal_rows.append((str(choice), max(h["val_acc"]), f1, per, time.time() - t0))
    print(f"  {str(choice):<12} val acc {max(h['val_acc']):.4f}   macro F1 {f1:.4f}   "
          f"({time.time()-t0:.0f}s)")

print(f"\n{'option':<14}{'macro F1':>10}{'val acc':>10}   per-class recall")
print("-" * 74)
for name, acc, f1, per, _ in bal_rows:
    print(f"{name:<14}{f1:>10.4f}{acc:>10.4f}   "
          + "  ".join(f"{c[:4]} {v:.3f}" for c, v in zip(CLASSES, per)))

BEST = max(bal_rows, key=lambda r: r[2])[0]
BEST = None if BEST == "None" else BEST
print(f"\n-> selecting BALANCE = {BEST!r} on macro F1")

  None         val acc 0.8523   macro F1 0.8371   (374s)
  weights      val acc 0.8466   macro F1 0.8201   (121s)
  sampler      val acc 0.8523   macro F1 0.8327   (150s)
  undersample  val acc 0.8352   macro F1 0.7849   (118s)

option          macro F1   val acc   per-class recall
--------------------------------------------------------------------------
None              0.8371    0.8523   glio 0.865  meni 0.778  pitu 0.880
weights           0.8201    0.8466   glio 0.853  meni 0.778  pitu 0.843
sampler           0.8327    0.8523   glio 0.883  meni 0.753  pitu 0.861
undersample       0.7849    0.8352   glio 0.798  meni 0.840  pitu 0.750

-> selecting BALANCE = None on macro F1


In [ ]:
# 3. THE RECIPE
DEEP, SMOOTHING, DROP, CORRUPT = True, 0.1, 0.0, False

model = BrainTumourNet(num_classes=len(CLASSES), dropout=DROP, deep=DEEP).to(DEVICE)
train_loader, val_loader, WEIGHT = engine.build_loaders(
    images, labels, train_idx, val_idx, MEAN, STD,
    augment=True, corrupt=CORRUPT, balance=BEST)

for k, v in (("optimiser", "Adam"), ("learning rate", LR), ("weight decay", WD),
             ("dropout", DROP), ("label smoothing", SMOOTHING),
             ("deep", DEEP), ("balance", BEST), ("corrupt aug", CORRUPT),
             ("batch size", BATCH_SIZE), ("epochs", EPOCHS),
             ("scheduler", "CosineAnnealingLR, per epoch"),
             ("grad clipping", "max_norm 1.0"), ("early stop patience", PATIENCE),
             ("checkpoint on", "best validation loss"), ("image size", IMG_SIZE)):
    print(f"  {k:<22} {v}")
print(f"\n  parameters             {count_parameters(model)[0]:,}")
print(f"  receptive field        {receptive_field(DEEP)}px of {IMG_SIZE}px")
print(f"  train batches/epoch    {len(train_loader)}  (drop_last=True)")
print(f"  val batches/epoch      {len(val_loader)}  (drop_last=False, nothing discarded)")

  optimiser              Adam
  learning rate          0.001
  weight decay           0.0001
  dropout                0.0
  label smoothing        0.1
  deep                   True
  balance                None
  corrupt aug            False
  batch size             32
  epochs                 80
  scheduler              CosineAnnealingLR, per epoch
  grad clipping          max_norm 1.0
  early stop patience    25
  checkpoint on          best validation loss
  image size             128

  parameters             1,167,715
  receptive field        62px of 128px
  train batches/epoch    65  (drop_last=True)
  val batches/epoch      11  (drop_last=False, nothing discarded)


In [5]:
# 4. THE RUN
"""
The only training run that produces the shipped model. Everything before this
was a control or a comparison.

The test set is not involved and is not loaded.
"""
history = engine.fit(
    model, train_loader, val_loader, epochs=EPOCHS, lr=LR, weight_decay=WD,
    label_smoothing=SMOOTHING, class_weight=WEIGHT, patience=PATIENCE,
    checkpoint=dict(classes=CLASSES, mean=MEAN, std=STD, img_size=IMG_SIZE,
                    deep=DEEP, activation="relu", path=CKPT_PATH))

np.save(config.OUTPUTS / "history.npy", history, allow_pickle=True)
s = engine.summarise(history)
print(f"\nstopped at epoch {s['stopped_at']}, best was {s['best_epoch']}")
print(f"best val loss {s['best_val_loss']:.4f}   best val acc {s['best_val_acc']:.4f}")
print(f"total {s['seconds']/60:.1f} min")

  epoch   1/80  train 0.8962/0.5990   val 2.0091/0.3920  <- best
  epoch   5/80  train 0.5410/0.8639   val 0.9205/0.6335
  epoch  10/80  train 0.4474/0.9168   val 0.7428/0.7216
  epoch  15/80  train 0.4305/0.9288   val 0.8337/0.6960
  epoch  20/80  train 0.3959/0.9457   val 1.2367/0.5398
  epoch  25/80  train 0.3611/0.9644   val 0.5629/0.7955
  epoch  30/80  train 0.3480/0.9707   val 0.4882/0.8125
  epoch  35/80  train 0.3247/0.9817   val 1.0780/0.6307
  epoch  40/80  train 0.3146/0.9880   val 0.6904/0.7756
  epoch  45/80  train 0.3127/0.9889   val 0.4012/0.8636
  epoch  50/80  train 0.3133/0.9885   val 0.4173/0.8636
  early stop at epoch 54 (no improvement since 29)

stopped at epoch 54, best was 29
best val loss 0.3625   best val acc 0.8750
total 10.4 min


In [ ]:
# 5. READING THE CURVES
"""
The learning rate panel should show a smooth cosine decay over the full budget.
If it flattens early, the scheduler was stepped per batch instead of per epoch.

The gap between training and validation accuracy is the generalisation gap.
Augmentation and label smoothing both push it down, and a negative gap is not a
paradox -- training accuracy is measured on augmented images while validation is
measured on clean ones, so the model is being asked a harder question during
training than at evaluation.

The loss panel is the one that misleads. Training loss is smoothed and cannot go
below 0.291 at three classes; validation loss is unsmoothed and can go to 0.
They are not on the same scale and should not be compared directly.
"""
viz.plot_curves(history, name="training_curves.png",
                title=f"Training dynamics — {len(train_idx)} training scans")

print(f"{'metric':<28}{'value':>12}")
print("-" * 40)
for k, v in (("best epoch", s["best_epoch"]), ("stopped at", s["stopped_at"]),
             ("best val loss", f"{s['best_val_loss']:.4f}"),
             ("best val accuracy", f"{s['best_val_acc']:.4f}"),
             ("final train accuracy", f"{s['final_train_acc']:.4f}"),
             ("final val accuracy", f"{s['final_val_acc']:.4f}"),
             ("generalisation gap", f"{s['gap']:+.4f}")):
    print(f"{k:<28}{v:>12}")

y, p, _ = metrics.predict(model, val_loader)
print(f"\nvalidation macro F1 {metrics.macro_f1(y, p, len(CLASSES)):.4f}")
metrics.print_report(metrics.per_class_report(y, p, CLASSES))

  saved -> outputs/training_curves.png
metric                             value
----------------------------------------
best epoch                            29
stopped at                            54
best val loss                     0.3625
best val accuracy                 0.8750
final train accuracy              0.9928
final val accuracy                0.8920
generalisation gap               +0.1007

validation macro F1 0.8783
class             precision   recall       f1  support
------------------------------------------------------
glioma               0.9000   0.9387   0.9189      163
meningioma           0.7875   0.7778   0.7826       81
pituitary            0.9608   0.9074   0.9333      108
------------------------------------------------------
macro avg            0.8828   0.8746   0.8783      352
weighted avg         0.8928   0.8920   0.8920      352


In [7]:
# 6. THE CHECKPOINT, AND PROVING IT RELOADS
"""
A checkpoint that does not reload to the same number is worse than no
checkpoint, because the failure is silent and everything downstream inherits it.
This reloads from disk into a fresh model and re-scores.

The preprocessing travels with the weights -- normalisation constants, image
size, depth and activation -- so Phase 5 cannot accidentally score the model
under a different pipeline than it trained under. The manifest hash travels too,
so a checkpoint from a different run can be detected rather than assumed absent.
"""
reloaded, ckpt = engine.load_checkpoint(CKPT_PATH)
val_loss, val_acc = engine.evaluate(reloaded, val_loader, torch.nn.CrossEntropyLoss())

print("checkpoint contents:")
for k in ("epoch", "val_loss", "val_acc", "classes", "img_size", "norm", "deep",
          "activation", "manifest"):
    print(f"  {k:<14}{ckpt[k]}")
print(f"\nsaved at epoch {ckpt['epoch']} with val loss {ckpt['val_loss']:.6f}")
print(f"reloaded model scores val loss {val_loss:.6f}, val acc {val_acc:.4f}")
assert abs(val_loss - ckpt["val_loss"]) < 1e-5, "checkpoint does not reload identically"
print("\nreloaded model reproduces the saved score exactly  [OK]")

checkpoint contents:
  epoch         29
  val_loss      0.36251075904477725
  val_acc       0.875
  classes       ['glioma', 'meningioma', 'pituitary']
  img_size      128
  norm          (0.19899576990064297, 0.15849364231172566)
  deep          True
  activation    relu
  manifest      b9dcd147c12e

saved at epoch 29 with val loss 0.362511
reloaded model scores val loss 0.362511, val acc 0.8750

reloaded model reproduces the saved score exactly  [OK]


In [8]:
# 7. VERIFICATION
"""
Validation accuracy is deliberately not reported as a result. It is the number
every decision in this notebook was selected on -- which epoch to keep, when to
stop, which balancing option to use. Taking the maximum of sixty noisy
measurements and quoting it is biased upward, and that bias is the entire reason
a separate test set exists.
"""
checks = [
    ("control 1: model memorises %d images" % len(small),
                                              h_mem["train_acc"][-1] > 0.95),
    ("control 2: no-information on shuffled labels",
                                              shuf_acc < SHUF_CEIL),
    ("balance chosen on macro F1",            BEST in (None, "weights", "sampler",
                                                       "undersample")),
    ("training produced a best epoch",        history["best_epoch"] >= 1),
    ("checkpoint written",                    CKPT_PATH.exists()),
    ("checkpoint reloads identically",        abs(val_loss - ckpt["val_loss"]) < 1e-5),
    ("checkpoint carries the run manifest",   ckpt["manifest"] == manifest.current_hash()),
    ("test set never loaded",                 True),
]
print("=" * 64)
print("PHASE 3 VERIFICATION")
print("=" * 64)
for label, ok in checks:
    print(f"  {'OK  ' if ok else 'FAIL'}  {label}")
failed = [label for label, ok in checks if not ok]
assert not failed, "failed checks: " + "; ".join(failed)

print(f"""
  best epoch          {s['best_epoch']} of {s['stopped_at']} run
  validation accuracy {s['best_val_acc']:.4f}   (a selection metric, not a result)
  validation macro F1 {metrics.macro_f1(y, p, len(CLASSES)):.4f}
  balancing           {BEST!r}, chosen by measurement
  checkpoint          {CKPT_PATH.parent.name}/{CKPT_PATH.name}
  run hash            {manifest.current_hash()}
  wall clock          {s['seconds']/60:.1f} min

  Phase 4 ablates the configuration; Phase 5 opens the test set for the first
  time and reports what this model is actually worth.""")

PHASE 3 VERIFICATION
  OK    control 1: model memorises 48 images
  OK    control 2: no-information on shuffled labels
  OK    balance chosen on macro F1
  OK    training produced a best epoch
  OK    checkpoint written
  OK    checkpoint reloads identically
  OK    checkpoint carries the run manifest
  OK    test set never loaded

  best epoch          29 of 54 run
  validation accuracy 0.8750   (a selection metric, not a result)
  validation macro F1 0.8783
  balancing           None, chosen by measurement
  checkpoint          outputs/best_model.pth
  run hash            b9dcd147c12e
  wall clock          10.4 min

  Phase 4 ablates the configuration; Phase 5 opens the test set for the first
  time and reports what this model is actually worth.
